In [ ]:
# --- Simple sanity check: prox comparison with missing labels ---

import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents) if (p/"src").is_dir())))

# Alignment models
from src.fosta import FoSTA as FoSTA_New
from src.fosta_icml import FoSTA as FoSTA_GT
from src.hiref.adaptive_HiRef import solve_surjection_hiref



import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Data
# -----------------------------
rng = np.random.default_rng(0)

X0, y = make_blobs(n_samples=120, centers=3, n_features=8, random_state=0)

X_a = StandardScaler().fit_transform(X0)

R = rng.normal(size=(8, 8))
U, _, Vt = np.linalg.svd(R, full_matrices=False)
R = U @ Vt

X_b = StandardScaler().fit_transform(X0 @ R + rng.normal(scale=0.25, size=X0.shape))

y_a = y.copy()
y_b = y.copy()

# remove 20% labels in B
idx = rng.choice(len(y_b), size=int(0.2 * len(y_b)), replace=False)
y_b[idx] = -1

# -----------------------------
# Fit models
# -----------------------------
gt = FoSTA_GT(random_state=0, verbose=0)
new = FoSTA_New(random_state=0, verbose=0)

gt.fit(X_a, X_b, y_a, y_b)
new.fit(X_a, X_b, y_a, y_b)

# -----------------------------
# Helpers
# -----------------------------
def to_dense(A):
    return A.toarray() if sparse.issparse(A) else A

def compare(A, B):
    A, B = to_dense(A), to_dense(B)
    D = A - B
    n = A.shape[0]

    diag = np.eye(n, dtype=bool)

    return {
        "diag_diff": np.mean(np.abs(D[diag])),
        "offdiag_diff": np.mean(np.abs(D[~diag])),
        "diag_corr": np.corrcoef(A[diag], B[diag])[0,1],
        "offdiag_corr": np.corrcoef(A[~diag], B[~diag])[0,1],
    }

# -----------------------------
# Results
# -----------------------------
df = pd.DataFrame([
    {"matrix": "prox_a", **compare(gt.prox_a, new.prox_a)},
    {"matrix": "prox_b", **compare(gt.prox_b, new.prox_b)},
])

display(df)

# Check Diagonal

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

def get_diag(A):
    if sparse.issparse(A):
        return A.diagonal()
    return np.diag(np.asarray(A))

def diag_stats(name, A):
    d = get_diag(A)
    return {
        "matrix": name,
        "min": d.min(),
        "max": d.max(),
        "mean": d.mean(),
        "std": d.std(),
    }

def diag_compare(name, A, B):
    dA = get_diag(A)
    dB = get_diag(B)
    diff = dA - dB
    return {
        "matrix": name,
        "linf_diff": np.max(np.abs(diff)),
        "mean_diff": np.mean(np.abs(diff)),
        "corr": np.corrcoef(dA, dB)[0, 1],
    }

display(pd.DataFrame([
    diag_stats("GT prox_a", gt.prox_a),
    diag_stats("New prox_a", new.prox_a),
    diag_stats("GT prox_b", gt.prox_b),
    diag_stats("New prox_b", new.prox_b),
]))

display(pd.DataFrame([
    diag_compare("prox_a", gt.prox_a, new.prox_a),
    diag_compare("prox_b", gt.prox_b, new.prox_b),
]))

print("GT prox_b diag:", get_diag(gt.prox_b)[:10])
print("New prox_b diag:", get_diag(new.prox_b)[:10])

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

W_gt = gt.W.toarray() if sparse.issparse(gt.W) else np.asarray(gt.W)
W_new = new.W.toarray() if sparse.issparse(new.W) else np.asarray(new.W)

D = W_gt - W_new

diag = np.eye(W_gt.shape[0], dtype=bool)
offdiag = ~diag

pd.DataFrame([{
    "shape_gt": W_gt.shape,
    "shape_new": W_new.shape,
    "max_abs_diff": np.max(np.abs(D)),
    "mean_abs_diff": np.mean(np.abs(D)),
    "fro_rel_diff": np.linalg.norm(D) / np.linalg.norm(W_gt),
    "diag_mean_abs_diff": np.mean(np.abs(D[diag])),
    "offdiag_mean_abs_diff": np.mean(np.abs(D[offdiag])),
    "corr_flat": np.corrcoef(W_gt.ravel(), W_new.ravel())[0, 1],
}])

In [ ]:
import numpy as np
import pandas as pd

rows = []

for name, A, B in [
    ("post_a", gt.post_a, new.post_a),
    ("post_b", gt.post_b, new.post_b),
]:
    A = np.asarray(A)
    B = np.asarray(B)
    D = A - B

    rows.append({
        "matrix": name,
        "shape_gt": A.shape,
        "shape_new": B.shape,
        "max_abs_diff": np.max(np.abs(D)),
        "mean_abs_diff": np.mean(np.abs(D)),
        "fro_rel_diff": np.linalg.norm(D) / max(np.linalg.norm(A), 1e-12),
        "corr_flat": np.corrcoef(A.ravel(), B.ravel())[0, 1],
        "sum_gt": A.sum(),
        "sum_new": B.sum(),
    })

pd.DataFrame(rows)

# Coupling T seems sensitive to tiny differences in post_a / post_b ?? Weird

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

T_gt = gt.T_sparse.toarray() if sparse.issparse(gt.T_sparse) else np.asarray(gt.T_sparse)
T_new = new.T_sparse.toarray() if sparse.issparse(new.T_sparse) else np.asarray(new.T_sparse)

D = T_gt - T_new

pd.DataFrame([{
    "shape_gt": T_gt.shape,
    "shape_new": T_new.shape,
    "max_abs_diff": np.max(np.abs(D)),
    "mean_abs_diff": np.mean(np.abs(D)),
    "fro_rel_diff": np.linalg.norm(D) / max(np.linalg.norm(T_gt), 1e-12),
    "corr_flat": np.corrcoef(T_gt.ravel(), T_new.ravel())[0, 1],
    "sum_gt": T_gt.sum(),
    "sum_new": T_new.sum(),
}])

# Check that coupling is the same for identical input

In [ ]:
T1 = solve_surjection_hiref(gt.post_a, gt.post_b, verbose=0, random_state=0)
T2 = solve_surjection_hiref(gt.post_a, gt.post_b, verbose=0, random_state=0)

T1 = T1.toarray() if sparse.issparse(T1) else np.asarray(T1)
T2 = T2.toarray() if sparse.issparse(T2) else np.asarray(T2)

D = T1 - T2

pd.DataFrame([{
    "max_abs_diff": np.max(np.abs(D)),
    "mean_abs_diff": np.mean(np.abs(D)),
    "fro_rel_diff": np.linalg.norm(D) / max(np.linalg.norm(T1), 1e-12),
    "corr_flat": np.corrcoef(T1.ravel(), T2.ravel())[0, 1],
    "sum_1": T1.sum(),
    "sum_2": T2.sum(),
}])